In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "rank_bm25"
])

import pandas as pd
import numpy as np
import re
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

DATA_PATH = "/kaggle/input/competitions/cohort-x-task-3/Task_3.xlsx"
ICD_PATH = "/kaggle/input/competitions/cohort-x-task-3/mimic-iv_icd-10_dict.xlsx"

MODEL_NAME = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"

TOP_K = 8

ALPHA = 0.72
BETA = 0.18
GAMMA = 0.10

KEEP_THRESHOLD = 0.91
ASSOCIATION_THRESHOLD = 0.73

BATCH_SIZE = 256

ABBREVIATIONS = {
    "htn": "hypertension",
    "dm": "diabetes mellitus",
    "ckd": "chronic kidney disease",
    "cad": "coronary artery disease",
    "copd": "chronic obstructive pulmonary disease",
    "hf": "heart failure",
    "mi": "myocardial infarction",
    "af": "atrial fibrillation",
    "aki": "acute kidney injury",
    "cva": "cerebrovascular accident",
    "uti": "urinary tract infection",
    "oa": "osteoarthritis",
    "ra": "rheumatoid arthritis",
    "pna": "pneumonia",
    "dka": "diabetic ketoacidosis",
    "sepsis": "systemic infection"
}

def load_data():
    xls = pd.ExcelFile(DATA_PATH)

    test = pd.read_excel(
        xls,
        sheet_name="Test"
    )

    icd = pd.read_excel(ICD_PATH)

    return test, icd

def normalize_text(text):
    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    expanded_words = []

    for word in text.split():
        expanded_words.extend(
            ABBREVIATIONS.get(word, word).split()
        )

    return " ".join(expanded_words)

def preprocess(test, icd):
    test["processed"] = test["Condition"].apply(
        normalize_text
    )

    icd["processed"] = icd["long_title"].apply(
        normalize_text
    )

    return test, icd

def load_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SentenceTransformer(
        MODEL_NAME,
        device=device
    )

    return model

def encode_texts(model, texts):
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return embeddings.astype(np.float32)

def build_tfidf(icd_texts):
    vectorizer = TfidfVectorizer(
        max_features=25000,
        ngram_range=(1, 3),
        stop_words="english",
        sublinear_tf=True
    )

    icd_matrix = vectorizer.fit_transform(
        icd_texts
    )

    return vectorizer, icd_matrix

def build_bm25(icd_texts):
    tokenized_texts = [
        text.split()
        for text in icd_texts
    ]

    return BM25Okapi(tokenized_texts)

def normalize_scores(scores):
    scores = np.array(scores)

    return (
        scores - scores.min()
    ) / (
        scores.max() - scores.min() + 1e-9
    )

def retrieve_candidates(
    query,
    query_embedding,
    icd_embeddings,
    tfidf_vectorizer,
    icd_tfidf,
    bm25
):
    semantic_scores = cosine_similarity(
        query_embedding.reshape(1, -1),
        icd_embeddings
    )[0]

    semantic_scores = normalize_scores(
        semantic_scores
    )

    tfidf_query = tfidf_vectorizer.transform(
        [query]
    )

    tfidf_scores = cosine_similarity(
        tfidf_query,
        icd_tfidf
    )[0]

    tfidf_scores = normalize_scores(
        tfidf_scores
    )

    bm25_scores = bm25.get_scores(
        query.split()
    )

    bm25_scores = normalize_scores(
        bm25_scores
    )

    combined_scores = (
        ALPHA * semantic_scores +
        BETA * tfidf_scores +
        GAMMA * bm25_scores
    )

    top_indices = np.argsort(
        combined_scores
    )[-TOP_K:][::-1]

    top_scores = combined_scores[
        top_indices
    ]

    return top_indices, top_scores

def assign_labels(
    top_codes,
    top_scores
):
    keep = []
    association = []
    diff = []

    max_score = top_scores[0]

    for code, score in zip(
        top_codes,
        top_scores
    ):
        relative_score = score / (
            max_score + 1e-9
        )

        if relative_score >= KEEP_THRESHOLD:
            keep.append(code)

        elif relative_score >= ASSOCIATION_THRESHOLD:
            association.append(code)

        else:
            diff.append(code)

    if len(keep) == 0:
        keep.append(top_codes[0])

    format_output = lambda x: (
        "; ".join(x)
        if len(x) > 0
        else "Not Applicable"
    )

    return (
        format_output(keep),
        format_output(association),
        format_output(diff)
    )

def predict(
    test,
    icd,
    test_embeddings,
    icd_embeddings,
    tfidf_vectorizer,
    icd_tfidf,
    bm25
):
    predictions = []

    for idx, row in test.iterrows():
        query = row["processed"]

        top_indices, top_scores = retrieve_candidates(
            query=query,
            query_embedding=test_embeddings[idx],
            icd_embeddings=icd_embeddings,
            tfidf_vectorizer=tfidf_vectorizer,
            icd_tfidf=icd_tfidf,
            bm25=bm25
        )

        top_codes = icd.iloc[
            top_indices
        ]["icd_code"].tolist()

        keep, association, diff = assign_labels(
            top_codes,
            top_scores
        )

        predictions.append({
            "Condition": row["Condition"],
            "KEEP": keep,
            "ASSOCIATION": association,
            "DIFF": diff
        })

    return pd.DataFrame(predictions)

def main():
    print("Loading datasets...")

    test, icd = load_data()

    print("Preprocessing text...")

    test, icd = preprocess(
        test,
        icd
    )

    print("Loading BioBERT model...")

    model = load_model()

    if torch.cuda.is_available():
        print(
            f"GPU: {torch.cuda.get_device_name(0)}"
        )

    print("Encoding test conditions...")

    test_embeddings = encode_texts(
        model,
        test["processed"].tolist()
    )

    print("Encoding ICD titles...")

    icd_embeddings = encode_texts(
        model,
        icd["processed"].tolist()
    )

    print("Building TF-IDF index...")

    tfidf_vectorizer, icd_tfidf = build_tfidf(
        icd["processed"].tolist()
    )

    print("Building BM25 index...")

    bm25 = build_bm25(
        icd["processed"].tolist()
    )

    print("Generating predictions...")

    submission = predict(
        test=test,
        icd=icd,
        test_embeddings=test_embeddings,
        icd_embeddings=icd_embeddings,
        tfidf_vectorizer=tfidf_vectorizer,
        icd_tfidf=icd_tfidf,
        bm25=bm25
    )

    submission.to_csv(
        "submission.csv",
        index=False
    )

    print("Submission saved.")

    return submission

submission = main()

submission.head()